# Spatial cross-validation setup

Define the cross-validation folds used to evaluate the model (Phase H): spatial blocks of
stations (KMeans on station coordinates), `GroupKFold` splits over those blocks, and a random
`KFold` baseline to compare against. Blocks are assigned per station, not per station-month, so
every month of a station stays in the same fold. The blocks are plotted on a map to check that
they are geographically coherent before any model is trained.

In [35]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Imports

In [34]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, KFold
import yaml
import seaborn as sns
import folium

import monitoring_deserts.models.spatial_cv as spatial_cv

## Configuration

In [36]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "cross_validation"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
MAPS_DIR = Path(gen_config["output_dir"]["maps"]) / "cross_validation"
PLOTS_DIR = Path(gen_config["output_dir"]["plots"]) / "cross_validation"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

EQUAL_AREA_CRS = gen_config["crs"]["equal_area"] # for distances in meters (buffers), not for storage

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())

# Cross-validation settings
RANDOM_STATE = 42
N_INIT = 15
STATIONS_PER_BLOCK = 10
N_FOLDS = 8

# Station-month table from notebook 7 (cleaned, with the log-transformed target)
analysis_df = pd.read_parquet(PROCESSED_DATA_DIR / "master_station_df.parquet")

# The parquet file loads month_of_year back as a plain integer, so restore the categorical
# dtype notebook 7 gave it (XGBoost only treats it as categorical with enable_categorical=True)
analysis_df["month_of_year"] = pd.Categorical(analysis_df["month_of_year"], categories=range(1, 13))

## Step 1 - Define the clusters for spatial cross validation

In [38]:
## Cluster the stations into spatial blocks (KMeans on the coordinates, roughly STATIONS_PER_BLOCK
## stations each) and give every row of the table the block of its station
analysis_df = spatial_cv.cluster_stations(analysis_df,
                                          stations_per_block=STATIONS_PER_BLOCK,
                                          n_init=N_INIT,
                                          random_state=RANDOM_STATE)

# Define the folds using GroupKFold: whole blocks are packed into N_FOLDS folds
gkf_splits = spatial_cv.group_k_fold(analysis_df, n_folds=N_FOLDS)


# Visualize the different spatial blocks 
stations = analysis_df.drop_duplicates("station_id")[["station_id", "longitude", "latitude", "spatial_block_id"]]
stations_gdf = gpd.GeoDataFrame(stations,
                                geometry=gpd.points_from_xy(stations.longitude, stations.latitude),
                                crs="EPSG:4326")
stations_gdf[["station_id", "spatial_block_id", "geometry"]].explore("spatial_block_id", marker_kwds={"radius": 5})


## Step 2 - Random KFold baseline

Ordinary k-fold over shuffled station-month rows, with the same number of folds as the spatial scheme so the two compare fold for fold. A station's other months mostly end up in the training set, which is the leakage the spatial folds remove. Phase J scores both with the same model and reports the two numbers side by side.

In [39]:
# Ordinary random k-fold over the station-month rows: same number of folds as the spatial scheme,
# but the rows are shuffled, so a station's other months are usually in the training set
random_splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(analysis_df))

## Next steps for this notebook

1. **Run the cross-validation.** Fit a baseline Random Forest on each fold of `random_splits` and of `gkf_splits`, and compare R², RMSE and MAE (on the µg/m³ scale, from pooled out-of-fold predictions). The plan's `run_cv` helper does this (Phase I step 2; it is not in the repo yet). Use the Phase I features, without raw latitude/longitude. The gap between the random and the spatial score is the headline finding of Phase H.
2. **Distance check.** Add the diagnostic behind plan v13's Appendix D.8 as a cell, so its numbers can be reproduced: for `gkf_splits` and `random_splits`, the distance from each held-out station to its nearest training station, next to the distance from populated grid cells to their nearest station.
3. **Buffered cross-check: decide.** On these blocks a 10-25 km dead zone removes nothing, so only build it if a semivariogram of log-PM2.5 shows spatial dependence beyond about 50 km (Appendix D.15). It is not implemented in `spatial_cv.py` yet.
4. **Blocks and row order.** The blocks depend on the table's row order, so decide whether to sort the stations by `station_id` before clustering, and save the station-to-block table so Phases I-J reuse exactly the same blocks.
5. **Metadata.** Write `configs/cross_validation_metadata.yaml`: stations per block, `n_init`, random state, number of blocks and folds, the fold sizes of both schemes, and the cross-validation scores from step 1.
6. **Then Phases I-J.** Tune XGBoost with `RandomizedSearchCV`, passing `gkf_splits` as `cv`, and set `enable_categorical=True`, because `month_of_year` is categorical. Then the evaluation and interpretation in Phase J.